# Staffing walkthrough

Demand is fitted by weekday and hour, phone and chat headcount comes from Erlang C, email comes from a workload rule, and PuLP builds an 8-hour shift roster. SimPy then checks both the baseline roster and the solved roster, with abandonment included.

Run this notebook from the project root. The first code cell reruns the pipeline only when `data/` is missing extracts.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import gen_data
import fit_demand
import erlang
import optimise
import sim

needed = [
    "contacts.csv",
    "demand_forecast.csv",
    "staffing_requirements.csv",
    "optimal_shifts.csv",
    "coverage.csv",
    "cost_summary.csv",
    "sim_summary.csv",
]
if any(not (erlang.DATA / name).exists() for name in needed):
    gen_data.main()
    fit_demand.main()
    erlang.main()
    optimise.main()
    sim.main()
else:
    print("Extracts already present. Delete data/*.csv to rebuild.")


## Fitted demand against the known arrival rate

The generator writes the true hourly rate. The fit only sees the realised contacts, so the error is sampling noise from four weeks.

In [ ]:
import pandas as pd

forecast = pd.read_csv(erlang.DATA / "demand_forecast.csv")
truth = pd.read_csv(erlang.DATA / "true_hourly_rates.csv")
check = forecast.merge(truth, on=["channel", "dow", "hour"])
mae = (
    check.assign(abs_error=lambda frame: (frame["avg_arrivals"] - frame["arrival_rate"]).abs())
    .groupby("channel")["abs_error"]
    .mean()
    .round(2)
)
print("Mean absolute error, contacts per hour")
print(mae.to_string())
phone_monday = forecast[(forecast.channel == "phone") & (forecast.dow == 0)][
    ["hour", "avg_arrivals", "aht_sec", "erlangs"]
]
phone_monday


## Requirement, baseline, and optimal roster

`logged_in_required` is the Erlang C (or workload) pool. `rostered_required` grosses that up for 30% shrinkage. The baseline is a fixed weekly template. Hours below the requirement are where that template will miss the service level.

In [ ]:
from IPython.display import display

coverage = pd.read_csv(erlang.DATA / "coverage.csv")
costs = pd.read_csv(erlang.DATA / "cost_summary.csv")
display(costs)

monday = coverage[(coverage.channel == "phone") & (coverage.dow == 0)][
    [
        "hour",
        "avg_arrivals",
        "erlangs",
        "logged_in_required",
        "baseline_logged_in",
        "optimal_logged_in",
    ]
]
monday


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(erlang.DATA / "plots" / "phone_monday_coverage.png")))


## Simulation check

Both scenarios use the same fitted demand and the same random seeds. The queue includes abandonment, which Erlang C ignores, so treat the simulated service level as the operational result and the Erlang figure as the planning target.

In [ ]:
summary = pd.read_csv(erlang.DATA / "sim_summary.csv")
summary[
    [
        "channel",
        "scenario",
        "sla_pct",
        "sla_target_pct",
        "asa_sec",
        "abandon_pct",
        "occupancy",
        "offered",
    ]
].round(2)
